In [2]:
# Step 0: Build a "non-skin" garbage dataset from CIFAR-10
import os, shutil, numpy as np
from PIL import Image
import tensorflow as tf
from pathlib import Path

# --- Paths (use absolute to avoid CWD issues) ---
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data")
NON_SKIN_DIR = ROOT / "non_skin"
NUM_IMAGES = 3000            # ~balance with your skin data
TARGET_SIZE = (224, 224)     # matches your models

# Fresh folder
if NON_SKIN_DIR.exists():
    shutil.rmtree(NON_SKIN_DIR)
NON_SKIN_DIR.mkdir(parents=True, exist_ok=True)
print(f"📂 Created directory: {NON_SKIN_DIR}")

# Download CIFAR-10 (cached by Keras after first run)
print("⬇️ Downloading CIFAR-10 to use as 'Non-Skin' examples...")
(x_train, _), (x_test, _) = tf.keras.datasets.cifar10.load_data()
all_images = np.concatenate([x_train, x_test], axis=0)

# Sample without replacement
rng = np.random.default_rng(seed=42)
idx = rng.choice(len(all_images), size=NUM_IMAGES, replace=False)

# Save resized JPEGs
saved = 0
for i in idx:
    arr = all_images[i]
    img = Image.fromarray(arr)
    # NEAREST keeps pixelation (acts like low-quality proxy); BILINEAR is also fine
    img = img.resize(TARGET_SIZE, Image.NEAREST)
    img.save(NON_SKIN_DIR / f"noise_{i}.jpg", quality=90)
    saved += 1

print(f"✅ Saved {saved} non-skin images.")
print("Sample file:", sorted(os.listdir(NON_SKIN_DIR))[:5])


2025-12-15 13:31:27.208703: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 13:31:27.285690: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-15 13:31:27.320108: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-15 13:31:27.332570: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-15 13:31:27.368834: I tensorflow/core/platform/cpu_feature_guar

📂 Created directory: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data/non_skin
⬇️ Downloading CIFAR-10 to use as 'Non-Skin' examples...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
✅ Saved 3000 non-skin images.
Sample file: ['noise_10013.jpg', 'noise_10026.jpg', 'noise_10047.jpg', 'noise_10097.jpg', 'noise_10105.jpg']


In [3]:
# Step 1 — Build Gatekeeper CSVs
from pathlib import Path
import pandas as pd, numpy as np, os, random
from PIL import Image

# --- Paths ---
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data")
PAD_IMAGES = ROOT / "PAD-UFES" / "Images"
DERMNET_CANDIDATES = [
    ROOT / "dermnet",
    Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/data/dermnet"),
]
NON_SKIN_DIR = ROOT / "non_skin"     # created in Step 0
BG_DIR = ROOT / "skin_bg"            # will be created if missing

assert PAD_IMAGES.exists(), f"Missing PAD images at: {PAD_IMAGES}"
assert NON_SKIN_DIR.exists() and any(NON_SKIN_DIR.glob('*.jpg')), "Non-skin set not found."

BG_DIR.mkdir(parents=True, exist_ok=True)
DERMNET = next((p for p in DERMNET_CANDIDATES if p.exists()), None)

def list_images(root: Path):
    exts = {".jpg",".jpeg",".png",".bmp",".webp"}
    return [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in exts]

# --- Abnormal sources (PAD + DermNet if present) ---
abn_paths = list_images(PAD_IMAGES)
if DERMNET:
    abn_paths += list_images(DERMNET)
print(f"Abnormal images found: {len(abn_paths)} (PAD + {'DermNet' if DERMNET else 'no DermNet'})")

# --- Normal sources: Non-skin + background skin patches ---
non_skin = list_images(NON_SKIN_DIR)
print(f"Non-skin images: {len(non_skin)}")

def make_bg_patches(src_paths, out_dir, per_image=1, frac=0.45, max_total=3000, seed=42):
    random.seed(seed)
    saved = 0
    if any(out_dir.glob("*.jpg")):
        # reuse existing
        return len(list_images(out_dir))
    for p in src_paths:
        if saved >= max_total: break
        try:
            img = Image.open(p).convert("RGB")
            w, h = img.size
            cw, ch = max(64, int(w*frac)), max(64, int(h*frac))
            corners = [(0,0),(w-cw,0),(0,h-ch),(w-cw,h-ch)]
            for k in range(per_image):
                if saved >= max_total: break
                x, y = random.choice(corners)
                crop = img.crop((x,y,x+cw,y+ch)).resize((224,224), Image.BILINEAR)
                crop.save(out_dir / f"bg_{p.stem}_{k}.jpg", quality=90)
                saved += 1
        except Exception:
            pass
    return saved

bg_created = make_bg_patches(abn_paths, BG_DIR, per_image=1, frac=0.45, max_total=3000)
bg_paths = list_images(BG_DIR)
print(f"Background skin patches on disk: {len(bg_paths)} (newly created: {bg_created})")

# --- Combine & label (0=normal, 1=abnormal) ---
normal_paths = non_skin + bg_paths
labels_norm  = [0] * len(normal_paths)
labels_abn   = [1] * len(abn_paths)

paths  = normal_paths + abn_paths
labels = labels_norm + labels_abn
srcs   = (["non_skin"]*len(non_skin) + ["skin_bg"]*len(bg_paths) +
          ["pad_or_dermnet"]*len(abn_paths))

df = pd.DataFrame({
    "image_fullpath": [str(p) for p in paths],
    "image_relpath":  [str(p) for p in paths],   # keep absolute to avoid CWD issues
    "gate_label":     labels,
    "source":         srcs
})

print("Overall distribution:", df["gate_label"].value_counts().rename({0:"normal",1:"abnormal"}).to_dict())

# --- Stratified 80/20 split ---
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
idx_tr, idx_va = next(sss.split(df, df["gate_label"]))
train_df = df.iloc[idx_tr].reset_index(drop=True)
val_df   = df.iloc[idx_va].reset_index(drop=True)

OUT_TRAIN = ROOT / "gatekeeper_train.csv"
OUT_VAL   = ROOT / "gatekeeper_val.csv"
train_df.to_csv(OUT_TRAIN, index=False)
val_df.to_csv(OUT_VAL, index=False)

print("Saved:", OUT_TRAIN, "|", len(train_df))
print("Saved:", OUT_VAL,   "|", len(val_df))
print("Train balance:", train_df['gate_label'].value_counts().rename({0:"normal",1:"abnormal"}).to_dict())
print("Val balance:",   val_df['gate_label'].value_counts().rename({0:"normal",1:"abnormal"}).to_dict())


Abnormal images found: 2298 (PAD + no DermNet)
Non-skin images: 3000
Background skin patches on disk: 2298 (newly created: 2298)
Overall distribution: {'normal': 5298, 'abnormal': 2298}
Saved: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data/gatekeeper_train.csv | 6076
Saved: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data/gatekeeper_val.csv | 1520
Train balance: {'normal': 4238, 'abnormal': 1838}
Val balance: {'normal': 1060, 'abnormal': 460}


In [4]:
# Step 2A — Model 1 (Gatekeeper) — Stage 1 training
import os, math, json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print("TF:", tf.__version__)

# ---------- Paths ----------
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data")
TRAIN_CSV = ROOT / "gatekeeper_train.csv"
VAL_CSV   = ROOT / "gatekeeper_val.csv"
ART_DIR   = Path("data/artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.exists() and VAL_CSV.exists(), "Gatekeeper CSVs not found."

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

# Labels: 0 = normal (non-skin + skin background), 1 = abnormal (real skin conditions)
n0 = int(train_df['gate_label'].value_counts().get(0, 0))
n1 = int(train_df['gate_label'].value_counts().get(1, 0))
N  = n0 + n1
# Inverse-frequency weights (sum ~1 each side)
class_weight = {0: N/(2*n0), 1: N/(2*n1)}
print("class_weight:", class_weight)

IMG_SIZE  = 224
BATCH     = 32
AUTOTUNE  = tf.data.AUTOTUNE

# ---------- tf.data ----------
def _load(path, label):
    file = tf.io.read_file(path)
    img  = tf.image.decode_image(file, channels=3, expand_animations=False)
    img  = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img  = tf.cast(img, tf.float32)  # keep 0–255; model handles rescaling
    return img, tf.cast(label, tf.float32)

def _aug(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.10)
    img = tf.image.random_contrast(img, 0.90, 1.10)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_ds(df, shuffle, augment):
    paths  = df['image_fullpath'].astype(str).values
    labels = df['gate_label'].astype('int32').values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(_aug, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, shuffle=True,  augment=True)
val_ds   = make_ds(val_df,   shuffle=False, augment=False)

# ---------- Model ----------
def build_gatekeeper():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_224")
    x = layers.Rescaling(1./127.5, offset=-1.0, name="mnv3_rescale")(inputs)

    backbone = tf.keras.applications.MobileNetV3Small(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        pooling="avg",
        weights="imagenet",
        alpha=1.0
    )
    backbone.trainable = False
    backbone._name = "backbone"  # for later lookup

    x = backbone(x, training=False)
    x = layers.Dropout(0.2, name="drop")(x)
    out = layers.Dense(1, activation="sigmoid", name="gate_output")(x)
    return models.Model(inputs, out, name="Gatekeeper_MNV3")

model = build_gatekeeper()
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
    tf.keras.metrics.AUC(name="auc"),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=METRICS
)

ckpt = callbacks.ModelCheckpoint(
    filepath=str(ART_DIR / "gatekeeper_stage1.keras"),
    monitor="val_auc", mode="max",
    save_best_only=True, verbose=1
)
es = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True, verbose=1)

print("Training Stage-1 (frozen backbone)…")
hist = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    class_weight=class_weight,
    callbacks=[ckpt, es],
    verbose=1
)

# ---------- Evaluation + threshold search ----------
print("\nEvaluating on val set…")
val_metrics = model.evaluate(val_ds, verbose=0, return_dict=True)
print("Val metrics:", {k: float(v) for k, v in val_metrics.items()})

# Collect raw probabilities for threshold tuning
y_true = []
y_prob = []
for batch_imgs, batch_labels in val_ds:
    p = model.predict(batch_imgs, verbose=0).squeeze()
    y_prob.extend(p.tolist())
    y_true.extend(batch_labels.numpy().tolist())

y_true = np.array(y_true, dtype=np.int32)
y_prob = np.array(y_prob, dtype=np.float32)

def thresh_table(y_true, y_prob, sens_target=0.95):
    grid = np.linspace(0.05, 0.95, 181)  # step 0.005
    rows = []
    for th in grid:
        y_pred = (y_prob >= th).astype(np.int32)  # 1 = abnormal
        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        sens = tp / max(tp + fn, 1)
        spec = tn / max(tn + fp, 1)
        prec = tp / max(tp + fp, 1)
        acc  = (tp + tn) / max(len(y_true), 1)
        rows.append(dict(th=float(th), tp=tp, fn=fn, fp=fp, tn=tn,
                         sens=float(sens), spec=float(spec),
                         prec=float(prec), acc=float(acc)))
    rows = [r for r in rows if r["sens"] >= sens_target]
    if not rows:
        return [], None
    # Prefer higher specificity, then higher precision
    rows = sorted(rows, key=lambda r: (r["spec"], r["prec"]), reverse=True)
    best = rows[0]
    return rows[:5], best

top5, best = thresh_table(y_true, y_prob, sens_target=0.95)
print(f"\nVal AUROC: {val_metrics['auc']:.4f}")
if best:
    print("Top 5 thresholds meeting sensitivity ≥ 0.95 (sorted by specificity then precision):")
    for r in top5:
        print(r)
    print("\nRecommended τ (abnormal=positive):", json.dumps(best, indent=2))
else:
    print("No threshold achieves sensitivity ≥ 0.95 yet. We will improve this in fine-tuning.")


TF: 2.17.1
class_weight: {0: 0.7168475696083058, 1: 1.6528835690968444}


I0000 00:00:1765806759.105712  153052 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1765806759.325635  153052 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1765806759.325685  153052 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1765806759.331818  153052 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1765806759.331862  153052 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:0

Training Stage-1 (frozen backbone)…
Epoch 1/12


I0000 00:00:1765806765.655593  156892 service.cc:146] XLA service 0x7275cc035f90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765806765.655681  156892 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 3070, Compute Capability 8.6
2025-12-15 13:52:45.915907: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-15 13:52:46.820789: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8906


  8/190 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.6195 - auc: 0.4845 - loss: 0.7893 - precision: 0.4604 - recall: 0.1431

I0000 00:00:1765806773.678100  156892 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.5516 - auc: 0.5479 - loss: 0.6957 - precision: 0.3462 - recall: 0.4974
Epoch 1: val_auc improved from -inf to 0.89203, saving model to data/artifacts/gatekeeper_stage1.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 32s 108ms/step - accuracy: 0.5517 - auc: 0.5481 - loss: 0.6956 - precision: 0.3463 - recall: 0.4977 - val_accuracy: 0.8197 - val_auc: 0.8920 - val_loss: 0.6660 - val_precision: 0.6431 - val_recall: 0.9087
Epoch 2/12
188/190 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.6382 - auc: 0.6941 - loss: 0.6546 - precision: 0.4343 - recall: 0.6308
Epoch 2: val_auc improved from 0.89203 to 0.89401, saving model to data/artifacts/gatekeeper_stage1.keras
190/190 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - accuracy: 0.6384 - auc: 0.6943 - loss: 0.6546 - precision: 0.4345 - recall: 0.6307 - val_accuracy: 0.8158 - val_auc: 0.8940 - val_loss: 0.6267 - val_precision: 0.7123 - val_recall: 0.6565
Epoch 3/12
190/190 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accur

2025-12-15 13:54:25.809254: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [6]:
# Step 2B — Gatekeeper fine-tune (robust backbone detection) + export
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, callbacks

print("TF:", tf.__version__)

# ---------- Paths ----------
ROOT   = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/skin-app/skin-app/data")
TRAIN  = ROOT / "gatekeeper_train.csv"
VAL    = ROOT / "gatekeeper_val.csv"
ART    = Path("data/artifacts")
ART.mkdir(parents=True, exist_ok=True)

STAGE1 = ART / "gatekeeper_stage1.keras"
STAGE2 = ART / "gatekeeper_stage2.keras"
TFLITE = ART / "gatekeeper_float32.tflite"
TH_JSON= ART / "gatekeeper_threshold.json"

assert TRAIN.exists() and VAL.exists(), "Missing Gatekeeper CSVs."
assert STAGE1.exists(), "Run Stage-1 first; gatekeeper_stage1.keras not found."

# ---------- Data ----------
IMG_SIZE = 224
BATCH    = 32
AUTOTUNE = tf.data.AUTOTUNE

train_df = pd.read_csv(TRAIN)
val_df   = pd.read_csv(VAL)

def _load(path, label):
    f  = tf.io.read_file(path)
    im = tf.image.decode_image(f, channels=3, expand_animations=False)
    im = tf.image.resize(im, (IMG_SIZE, IMG_SIZE))
    im = tf.cast(im, tf.float32)   # 0..255, model rescales internally
    return im, tf.cast(label, tf.float32)

def _aug(im, y):
    im = tf.image.random_flip_left_right(im)
    im = tf.image.random_brightness(im, 0.08)
    im = tf.image.random_contrast(im, 0.9, 1.1)
    im = tf.clip_by_value(im, 0.0, 255.0)
    return im, y

def make_ds(df, shuffle, augment):
    paths  = df["image_fullpath"].astype(str).values
    labels = df["gate_label"].astype("int32").values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(df), reshuffle_each_iteration=True)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(_aug, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_df, shuffle=True,  augment=True)
val_ds   = make_ds(val_df,   shuffle=False, augment=False)

# class weights (same recipe as Stage-1)
n0 = int(train_df['gate_label'].value_counts().get(0, 0))
n1 = int(train_df['gate_label'].value_counts().get(1, 0))
N  = n0 + n1
class_weight = {0: N/(2*n0), 1: N/(2*n1)}
print("class_weight:", class_weight)

# ---------- Load Stage-1 ----------
model = tf.keras.models.load_model(str(STAGE1))
model.summary(line_length=120)

# ---------- Robust backbone detection ----------
def find_backbone(m):
    # Preferred names
    for name in ("backbone", "MobileNetV3Small", "mobilenetv3small", "base_model"):
        try:
            return m.get_layer(name)
        except Exception:
            pass
    # Fallback: first submodel-ish layer
    candidates = [L for L in m.layers if hasattr(L, "layers")]
    assert candidates, "Could not locate backbone submodel in model."
    return candidates[0]

backbone = find_backbone(model)
print(f"Using backbone: {backbone.name} with {len(getattr(backbone, 'layers', []))} layers")

# ---------- Partial unfreeze (keep BatchNorm frozen) ----------
K = 60  # try 80 later if stable
# 1) freeze all
for L in getattr(backbone, "layers", []):
    L.trainable = False

# 2) unfreeze last K non-BN layers
bn_types = (layers.BatchNormalization,)
trainable_cnt = 0
for L in reversed(getattr(backbone, "layers", [])):
    if isinstance(L, bn_types):
        L.trainable = False
        continue
    if trainable_cnt < K:
        try:
            L.trainable = True
            trainable_cnt += 1
        except Exception:
            L.trainable = False
    else:
        L.trainable = False
print(f"Unfroze {trainable_cnt} non-BN layers in backbone.")

# ---------- Compile ----------
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
    tf.keras.metrics.AUC(name="auc"),
]
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="binary_crossentropy",
    metrics=METRICS,
)

ckpt = callbacks.ModelCheckpoint(
    filepath=str(STAGE2), monitor="val_auc", mode="max",
    save_best_only=True, verbose=1
)
es   = callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True, verbose=1)

print("\nTraining Stage-2 (partial unfreeze)…")
hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=[ckpt, es],
    verbose=1
)

# ---------- Evaluate + threshold search (sens ≥ 0.95) ----------
print("\nEvaluating on val set…")
val_metrics = model.evaluate(val_ds, verbose=0, return_dict=True)
print("Val metrics:", {k: float(v) for k, v in val_metrics.items()})

# gather scores
y_true, y_prob = [], []
for Xb, yb in val_ds:
    p = model.predict(Xb, verbose=0).squeeze()
    y_prob.extend(p.tolist())
    y_true.extend(yb.numpy().tolist())
y_true = np.array(y_true, dtype=np.int32)
y_prob = np.array(y_prob, dtype=np.float32)

def thresh_table(y_true, y_prob, sens_target=0.95):
    grid = np.linspace(0.05, 0.95, 181)  # step 0.005
    rows = []
    for th in grid:
        yp = (y_prob >= th).astype(np.int32)  # 1=abnormal
        tp = int(((y_true == 1) & (yp == 1)).sum())
        fn = int(((y_true == 1) & (yp == 0)).sum())
        fp = int(((y_true == 0) & (yp == 1)).sum())
        tn = int(((y_true == 0) & (yp == 0)).sum())
        sens = tp / max(tp + fn, 1)
        spec = tn / max(tn + fp, 1)
        prec = tp / max(tp + fp, 1)
        acc  = (tp + tn) / len(y_true)
        rows.append(dict(th=float(th), tp=tp, fn=fn, fp=fp, tn=tn,
                         sens=float(sens), spec=float(spec),
                         prec=float(prec), acc=float(acc)))
    rows = [r for r in rows if r["sens"] >= sens_target]
    if not rows:
        return [], None
    rows = sorted(rows, key=lambda r: (r["spec"], r["prec"]), reverse=True)
    return rows[:5], rows[0]

top5, best = thresh_table(y_true, y_prob, sens_target=0.95)
print(f"\nVal AUROC: {val_metrics['auc']:.4f}")
if best:
    print("Top 5 thresholds meeting sensitivity ≥ 0.95 (sorted by specificity then precision):")
    for r in top5:
        print(r)
    print("\nRecommended τ (abnormal=positive):", json.dumps(best, indent=2))
else:
    print("No threshold reaches sensitivity ≥ 0.95. Keeping τ=0.45 as a conservative default.")
    best = {"th": 0.45}

with open(TH_JSON, "w") as f:
    json.dump({"tau": float(best["th"])}, f, indent=2)
print(f"\nSaved threshold to {TH_JSON}")

# ---------- Export TFLite (float32 with DRQ) ----------
def export_tflite_float32(keras_model, out_path):
    conv = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic-range on weights
    tflite_model = conv.convert()
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    sz = os.path.getsize(out_path)/1024/1024
    print(f"Saved TFLite: {out_path} ({sz:.2f} MB)")

export_tflite_float32(model, str(TFLITE))

# ---------- Sanity preds ----------
sample_paths = val_df["image_fullpath"].astype(str).tolist()[:4]
x = []
for p in sample_paths:
    b = tf.io.read_file(p)
    im = tf.image.decode_image(b, channels=3, expand_animations=False)
    im = tf.image.resize(im, (IMG_SIZE, IMG_SIZE))
    x.append(tf.cast(im, tf.float32))
x = tf.stack(x, axis=0)  # [B,224,224,3]
probs = model.predict(x, verbose=0).squeeze()
print("\nSanity preds (first 4):", probs.tolist())

print("\nDone. Artifacts:")
print(" -", STAGE2)
print(" -", TFLITE)
print(" -", TH_JSON)


TF: 2.17.1
class_weight: {0: 0.7168475696083058, 1: 1.6528835690968444}


Model: "Gatekeeper_MNV3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━
┃ Layer (type)                                        ┃ Output Shape                           ┃               Para
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━
│ input_224 (InputLayer)                              │ (None, 224, 224, 3)                    │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ mnv3_rescale (Rescaling)                            │ (None, 224, 224, 3)                    │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ MobileNetV3Small (Functional)                       │ (None, 576)                            │               939,
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ drop (Dropout)                                      │ (None, 576)                            │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ gate_output (Dense)                                 │ (None, 1)                              │                   
└─────────────────────────────────────────────────────┴────────────────────────────────────────┴───────────────────

 Total params: 940,853 (3.59 MB)

 Trainable params: 577 (2.25 KB)

...


Val AUROC: 0.9357
Top 5 thresholds meeting sensitivity ≥ 0.95 (sorted by specificity then precision):
{'th': 0.22999999999999998, 'tp': 437, 'fn': 23, 'fp': 242, 'tn': 818, 'sens': 0.95, 'spec': 0.7716981132075472, 'prec': 0.6435935198821797, 'acc': 0.8256578947368421}
{'th': 0.22499999999999998, 'tp': 438, 'fn': 22, 'fp': 246, 'tn': 814, 'sens': 0.9521739130434783, 'spec': 0.7679245283018868, 'prec': 0.6403508771929824, 'acc': 0.8236842105263158}
{'th': 0.21999999999999997, 'tp': 438, 'fn': 22, 'fp': 248, 'tn': 812, 'sens': 0.9521739130434783, 'spec': 0.7660377358490567, 'prec': 0.6384839650145773, 'acc': 0.8223684210526315}
{'th': 0.21499999999999997, 'tp': 438, 'fn': 22, 'fp': 252, 'tn': 808, 'sens': 0.9521739130434783, 'spec': 0.7622641509433963, 'prec': 0.6347826086956522, 'acc': 0.8197368421052632}
{'th': 0.20999999999999996, 'tp': 440, 'fn': 20, 'fp': 257, 'tn': 803, 'sens': 0.9565217391304348, 'spec': 0.7575471698113208, 'prec': 0.6312769010043041, 'acc': 0.8177631578947369}



INFO:tensorflow:Assets written to: /tmp/tmpuvpntmx9/assets


Saved artifact at '/tmp/tmpuvpntmx9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_224')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  125853343654608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853343649424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853343649616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853343648656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853343648464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853344935248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853344933712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853344933520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853344933904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853344934672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  125853344933136: 

W0000 00:00:1765807395.902044  153052 tf_tfl_flatbuffer_helpers.cc:392] Ignored output_format.
W0000 00:00:1765807395.902087  153052 tf_tfl_flatbuffer_helpers.cc:395] Ignored drop_control_dependency.
2025-12-15 14:03:15.902687: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpuvpntmx9
2025-12-15 14:03:15.912177: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-12-15 14:03:15.912205: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpuvpntmx9
2025-12-15 14:03:16.001974: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2025-12-15 14:03:16.013345: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-12-15 14:03:16.487098: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpuvpntmx9
2025-12-15 14:03:16.599496: I tensorflow/cc/saved_model/loader.cc

Saved TFLite: data/artifacts/gatekeeper_float32.tflite (1.06 MB)

Sanity preds (first 4): [0.5480753779411316, 0.19013188779354095, 4.319615982240066e-06, 0.5562940835952759]

Done. Artifacts:
 - data/artifacts/gatekeeper_stage2.keras
 - data/artifacts/gatekeeper_float32.tflite
 - data/artifacts/gatekeeper_threshold.json
